In [1]:
from seismic import SeismicIndexDotVByte

In [3]:
json_input_file = "/data2/cosimorulli/sample.jsonl"
index = SeismicIndexDotVByte.build(json_input_file)


Building the index (standard u16)...
Configuration { pruning: GlobalThreshold { n_postings: 3500, max_fraction: 1.5 }, blocking: RandomKmeans { centroid_fraction: 0.1, min_cluster_size: 2, clustering_algorithm: RandomKmeansInvertedIndexApprox { doc_cut: 15 } }, summarization: EnergyPreserving { summary_energy: 0.4 }, knn: KnnConfiguration { nknn: 0, knn_path: None } }
Reading the collection..
Number of rows: 1000
Elapsed time to read the collection: 0 secs
Distributing and pruning postings: 0 secs
Number of posting lists: 13373
Avg posting list length: 9.12
Building summaries: 0 secs
Converting to DotVByte...


In [10]:
import numpy as np
import json

file_path = "/data2/cosimorulli/queries_anserini.tsv"

queries = []
with open(file_path, 'r') as f:
    for line in f:
        queries.append(json.loads(line))

MAX_TOKEN_LEN = 30
string_type  = f'U{MAX_TOKEN_LEN}'

queries_ids = np.array([q['id'] for q in queries], dtype=string_type)

query_components = []
query_values = []

for query in queries:
    vector = query['vector']
    query_components.append(np.array(list(vector.keys()), dtype=string_type))
    query_values.append(np.array(list(vector.values()), dtype=np.float32))
    
results = index.batch_search(
    queries_ids=queries_ids,
    query_components=query_components,
    query_values=query_values,
    k=10,
    query_cut=10,
    heap_factor=0.7,
    n_knn=0,
    sorted=True, 
    num_threads=1,
)

In [11]:
import ir_measures
import ir_datasets

# add your ir_dataset dataset string id below, e.g., "beir/quora/test"
ir_dataset_string = ""

ir_results = [ir_measures.ScoredDoc(query_id, doc_id, score) for r in results for (query_id, score, doc_id) in r]
qrels = ir_datasets.load(ir_dataset_string).qrels

Query: 1048585

Relevant passages:
1. [245] John Maynard Keynes was born at 7 Melville Road, Cambridge, England. His father was John Neville Keynes, an economics lecturer at Cambridge University. His mother was Florence Ada Brown, a successful author and a social reformer. His younger brother, Geoffrey Keynes (1887–1982) was a surgeon and bibliophile (book lover).

2. [668] Enochian alphabet. The Enochian alphabet first appeared during the 16th century. The Court Astrologer and Magician, Dr. John Dee (1527-1608), and his associate, Sir Edward Kelly (1555-1597) claimed that the alphabet and the Enochian language was transmitted to them by angels.nochian alphabet. The Enochian alphabet first appeared during the 16th century. The Court Astrologer and Magician, Dr. John Dee (1527-1608), and his associate, Sir Edward Kelly (1555-1597) claimed that the alphabet and the Enochian language was transmitted to them by angels.

3. [661] The term Enochian comes from Dee's assertion that the Biblica